In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


In [ ]:
# 1. Load Wine dataset into a Pandas DataFrame
raw_data = load_wine(as_frame=True)
df = raw_data.frame.copy()

# Drop the ground truth target column for unsupervised clustering
if 'target' in df.columns:
    df_unsupervised = df.drop(columns=['target'])
else:
    df_unsupervised = df.copy()

print(f"Dataset shape: {df_unsupervised.shape}")


In [ ]:
# 2. Display first and last 5 records
print("--- First 5 Records ---")
display(df_unsupervised.head(5))

print("\n--- Last 5 Records ---")
display(df_unsupervised.tail(5))


In [ ]:
# 3. Explore info, summary statistics, and data types
print("--- Dataset Info ---")
df_unsupervised.info()

print("\n--- Data Types ---")
print(df_unsupervised.dtypes)

print("\n--- Summary Statistics ---")
display(df_unsupervised.describe().T)


In [ ]:
# 4. Select numerical features
numerical_cols = df_unsupervised.select_dtypes(include=[np.number]).columns.tolist()
X_raw = df_unsupervised[numerical_cols].copy()

print(f"Selected {len(numerical_cols)} numerical features: {numerical_cols}")


In [ ]:
# 5. Standardize numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

df_scaled = pd.DataFrame(X_scaled, columns=numerical_cols)
display(df_scaled.head(3))


In [ ]:
# 6. Implement K-Means


In [ ]:
# 7. Apply K-Means with K = 2
kmeans_k2 = KMeans(n_clusters=2, random_state=42, n_init='auto')
labels_k2 = kmeans_k2.fit_predict(X_scaled)


In [ ]:
# 8. Apply K-Means with K = 3
kmeans_k3 = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels_k3 = kmeans_k3.fit_predict(X_scaled)

print(f"K=2 Inertia: {kmeans_k2.inertia_:.2f}")
print(f"K=3 Inertia: {kmeans_k3.inertia_:.2f}")


In [ ]:
# 9 & 10. Compare multiple K values and determine optimal K via Elbow Method
k_range = range(1, 11)
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, 'bo-', markersize=8)
plt.title('Task 10: Elbow Method For Optimal K', fontsize=12)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(k_range)
plt.show()


In [ ]:
# 11. Visualize clusters (using first 2 scaled features: alcohol vs malic_acid)


In [ ]:
# 12. Display cluster centroids
centroids_k3 = kmeans_k3.cluster_centers_

plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_k3, cmap='viridis', alpha=0.7, edgecolors='k')
plt.scatter(centroids_k3[:, 0], centroids_k3[:, 1], c='red', s=250, marker='X', label='Centroids')

plt.title('Task 11 & 12: K-Means (K=3) Clusters and Centroids')
plt.xlabel(f'{numerical_cols[0]} (Standardized)')
plt.ylabel(f'{numerical_cols[1]} (Standardized)')
plt.legend()
plt.show()

print("--- Cluster Centroids (First 2 dimensions shown) ---")
print(centroids_k3[:, :2])


In [ ]:
# 13. Assign cluster labels
df_clustered = df_unsupervised.copy()
df_clustered['KMeans_Cluster'] = labels_k3
display(df_clustered.head(5))


In [ ]:
# 14. Evaluate impact of scaling on K-Means
km_unscaled = KMeans(n_clusters=3, random_state=42, n_init='auto').fit(X_raw)
km_scaled = KMeans(n_clusters=3, random_state=42, n_init='auto').fit(X_scaled)

sil_unscaled = silhouette_score(X_raw, km_unscaled.labels_)
sil_scaled = silhouette_score(X_scaled, km_scaled.labels_)

print(f"Silhouette Score (Unscaled Data): {sil_unscaled:.4f}")
print(f"Silhouette Score (Scaled Data):   {sil_scaled:.4f}")
print("Note: Scaling prevents high-variance features (e.g., Proline) from dominating distance metrics.")


In [ ]:
# 15 & 16. Agglomerative Hierarchical Clustering dendrogram
plt.figure(figsize=(12, 5))
linkage_matrix = linkage(X_scaled, method='ward')
dendrogram(linkage_matrix, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=10)
plt.title('Task 16: Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Sample Index or Cluster Size')
plt.ylabel('Distance')
plt.show()


In [ ]:
# 17, 18, 19. Apply and compare Ward, Complete, and Average linkages
linkages = ['ward', 'complete', 'average']
hierarchical_results = {}

for link in linkages:
    model = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage=link)
    lbls = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, lbls)
    hierarchical_results[link] = (lbls, score)
    print(f"Linkage: {link:<10} | Silhouette Score: {score:.4f}")


In [ ]:
# 20. Scatter plot for Hierarchical Clustering (Ward)
ward_labels = hierarchical_results['ward'][0]

plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=ward_labels, cmap='plasma', alpha=0.7, edgecolors='k')
plt.title('Task 20: Agglomerative Clustering (Ward Linkage, K=3)')
plt.xlabel(f'{numerical_cols[0]} (Standardized)')
plt.ylabel(f'{numerical_cols[1]} (Standardized)')
plt.show()


In [ ]:
# 21. Quantitative comparison using Adjusted Rand Index and Cross-tabulation
ari = adjusted_rand_score(labels_k3, ward_labels)
print(f"Adjusted Rand Index between K-Means and Ward Hierarchical: {ari:.4f}\n")

print("Contingency Matrix (Cross-tabulation):")
ct = pd.crosstab(pd.Series(labels_k3, name='KMeans'), pd.Series(ward_labels, name='Ward_Hierarchical'))
display(ct)


In [ ]:
# 22 & 23. Apply PCA to reduce features to 2 components
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

df_pca_2d = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])


In [ ]:
# 24. Visualize 2D PCA projection
plt.figure(figsize=(8, 5))
plt.scatter(df_pca_2d['PC1'], df_pca_2d['PC2'], c=labels_k3, cmap='viridis', edgecolors='k', alpha=0.8)
plt.title('Task 24: 2D PCA Scatter Plot (Colored by K-Means Clusters)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()


In [ ]:
# 25 & 26. Explained variance and cumulative explained variance
pca_full = PCA().fit(X_scaled)
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

print("--- Task 25 & 26: Explained Variance ---")
for i, (ev, cv) in enumerate(zip(exp_var, cum_var), 1):
    print(f"PC{i:02d}: Explained = {ev*100:5.2f}% | Cumulative = {cv*100:6.2f}%")


In [ ]:
# 27. Compare dataset before and after dimensionality reduction
print("\n--- Task 27: Shape Comparison ---")
print(f"Original Feature Space Shape: {X_scaled.shape}")
print(f"PCA Reduced Space Shape:      {X_pca_2d.shape}")


In [ ]:
# 28. K-Means on PCA-transformed dataset (2 components)
kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels_pca = kmeans_pca.fit_predict(X_pca_2d)


In [ ]:
# 29. Compare performance
sil_full = silhouette_score(X_scaled, labels_k3)
sil_pca = silhouette_score(X_pca_2d, labels_pca)

print(f"Silhouette Score (All {X_scaled.shape[1]} Features): {sil_full:.4f}")
print(f"Silhouette Score (2 PCA Components):      {sil_pca:.4f}")


In [ ]:
# 30 & 31. Compute and compare Silhouette Scores for K = 2 to 6
silhouette_scores = []
k_test_range = range(2, 7)

for k in k_test_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    lbls = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, lbls)
    silhouette_scores.append(score)
    print(f"K = {k} | Silhouette Score: {score:.4f}")

plt.figure(figsize=(7, 4))
plt.bar(k_test_range, silhouette_scores, color='teal', edgecolor='black')
plt.title('Task 31: Silhouette Scores Across Values of K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
# 32. Cluster profile via mean and median of original feature values
cluster_summary = df_clustered.groupby('KMeans_Cluster').agg(['mean', 'std']).T
display(cluster_summary)


In [ ]:
# 33. Pair plot of key representative features
selected_features = ['alcohol', 'flavanoids', 'color_intensity', 'KMeans_Cluster']
sns.pairplot(df_clustered[selected_features], hue='KMeans_Cluster', palette='Set2')
plt.suptitle('Task 33: Cluster Distributions Across Key Features', y=1.02)
plt.show()


In [ ]:
# 34. Save results to CSV
output_path = 'clustered_wine_dataset.csv'
df_clustered.to_csv(output_path, index=False)
print(f"Saved clustered dataset successfully to '{output_path}'.")


In [ ]:
# 35. Compare K-Means and Hierarchical Clustering and summarize findings
summary_df = pd.DataFrame({
    'Metric': [
        'Silhouette Score',
        'Computational Complexity',
        'Shape Bias',
        'Interpretability'
    ],
    'K-Means': [
        f"{silhouette_score(X_scaled, labels_k3):.4f}",
        'O(n * k * i) - Highly Scalable',
        'Spherical / Convex clusters',
        'Centroid-based (easy mean profiling)'
    ],
    'Hierarchical (Ward)': [
        f"{silhouette_score(X_scaled, ward_labels):.4f}",
        'O(n^3) time, O(n^2) memory - Less scalable',
        'Minimizes variance (flexible)',
        'Dendrogram-based (nested taxonomies)'
    ]
})

display(summary_df)

print("""
Findings:
1. Feature Scaling: Crucial for distance-based clustering; features with large magnitudes otherwise skew partition boundaries.
2. Optimal K: Both the Elbow plot and Silhouette analysis identify K=3 as the natural partition for this feature space.
3. Dimensionality Reduction: Retaining 2 PCs captured ~55% of the total variance, allowing 2D visual separation while preserving core cluster boundaries.
4. Algorithm Fit: K-Means and Ward Hierarchical yield highly aligned cluster memberships (ARI > 0.75), verifying structural stability.
""")
